[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/01-shapes-and-linear-algebra.ipynb)

# Module 1: Shapes and Linear Algebra

Companion notebook to [Module 1](https://transformers-and-math.drmhse.com/modules/01-shapes-and-linear-algebra) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import numpy as np
import torch

## Part 1 — Shapes (section 1.1)
A scalar, a vector, a matrix and a tensor are arrays with 0, 1, 2 and 3 axes. `.shape` lists the length of each axis.

In [ ]:
x = torch.tensor(3.0)
v = torch.tensor([0.2, -1.7, 3.0, 0.5])
X = torch.tensor([[0.2, -1.7, 3.0, 0.5],
                  [1.1,  0.4, -0.3, 2.2],
                  [-0.6, 0.9, 1.5, 0.0]])
batch = torch.stack([X, X * 2])
for name, t in [('scalar', x), ('vector', v), ('matrix', X), ('tensor', batch)]:
    print(f'{name:7} axes={t.ndim}  shape={tuple(t.shape)}  numbers={t.numel()}')

# Indices count from 0 in code: X_23 in the course (row 2, column 3) is X[1, 2].
print('X_23 =', X[1, 2].item())

## Part 2 — Matrix multiplication by hand (section 1.2)
Entry $(i, j)$ of $AB$ is row $i$ of $A$ dotted with column $j$ of $B$. Write it with loops, then check against `@`.

In [ ]:
A = np.array([[1, 2, 0], [-1, 0, 3]], dtype=float)
B = np.array([[2, 1], [0, -1], [1, 4]], dtype=float)

def matmul(A, B):
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, 'inner sizes must agree'
    out = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            out[i, j] = sum(A[i, k] * B[k, j] for k in range(n))
    return out

print(matmul(A, B))
print('matches numpy:', np.allclose(matmul(A, B), A @ B))

# Each row of XW depends only on the matching row of X: tokens do not mix.
W = np.random.default_rng(0).normal(size=(4, 2))
X2 = X.numpy().copy()
X2[0] += 10                     # change token 1 only
print('rows that changed:', np.where(~np.isclose(X.numpy() @ W, X2 @ W).all(axis=1))[0])

## Part 3 — The matrix toolkit (section 1.3)

In [ ]:
M = np.array([[2., 1.], [1., 3.]])
print('transpose of a product reverses order:', np.allclose((A @ B).T, B.T @ A.T))
print('identity changes nothing:', np.allclose(M @ np.eye(2), M))
D = np.diag([2., 3.])
print('diag scales columns from the right:\n', M @ D)
print('inverse undoes:', np.allclose(np.linalg.inv(M) @ M, np.eye(2)))
u, w = np.array([1., 2., 3.]), np.array([4., 5.])
print('outer product shape:', np.outer(u, w).shape, ' rank:', np.linalg.matrix_rank(np.outer(u, w)))

## Part 4 — Dot products and cosine similarity (section 1.4)

In [ ]:
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

a, b = np.array([3., 4.]), np.array([4., 3.])
print('dot =', a @ b, ' cosine =', cosine(a, b))
print('scaling a changes the dot product but not the cosine:', (10 * a) @ b, cosine(10 * a, b))

## Part 5 — Embeddings are row lookups (section 1.5)
Multiplying a one-hot vector by $W_E$ picks out one row. `nn.Embedding` does the lookup directly.

In [ ]:
torch.manual_seed(0)
vocab, d_model = 10, 4
emb = torch.nn.Embedding(vocab, d_model)
i = 7
one_hot = torch.zeros(vocab); one_hot[i] = 1
print(torch.allclose(one_hot @ emb.weight, emb(torch.tensor(i))))

## Part 6 — Projections into Q, K and V (section 1.6)
The worked example from the module, then the rank of a projection.

In [ ]:
x1 = np.array([1., 0., 2., -1.]); x2 = np.array([0., 1., 1., 1.])
WQ = np.array([[1, 0], [0, 1], [1, -1], [0, 1]], dtype=float)
WK = np.array([[0, 1], [1, 0], [-1, 1], [1, 0]], dtype=float)
q1, k2 = x1 @ WQ, x2 @ WK
print('q1 =', q1, ' k2 =', k2, ' score =', q1 @ k2, ' raw x1·x2 =', x1 @ x2)

P = np.random.default_rng(1).normal(size=(768, 64))
print('rank of a 768×64 projection:', np.linalg.matrix_rank(P))

## Part 7 — How numbers are stored (section 1.7)

In [ ]:
for dtype in (torch.float32, torch.bfloat16, torch.float16):
    t = torch.zeros(50257, 768, dtype=dtype)
    print(f'{str(dtype):15} {t.element_size()} bytes/number  →  {t.numel() * t.element_size() / 1e6:.0f} MB')

## Your turn
1. Compute entry $(2, 1)$ of $AB$ for $A = \begin{bmatrix} 1 & 0 & 2 \\ 3 & -1 & 1 \end{bmatrix}$, $B = \begin{bmatrix} 4 & 0 \\ 2 & 5 \\ -1 & 1 \end{bmatrix}$ with your `matmul`, and by hand.
2. Build a 5 × 3 matrix of rank 2 (hint: a sum of two outer products) and check it with `np.linalg.matrix_rank`.
3. How many MB does a 32,000 × 4,096 embedding table take in bf16?

<details><summary>Answers</summary>

1. 9. 2. `np.outer(a, b) + np.outer(c, d)` for random `a, c` of length 5 and `b, d` of length 3. 3. 262 MB.
</details>